In [ ]:
import pandas as pd 
import datetime
JOURNAL_CODES = ['HA', 'CA', 'BC', 'AN']
fournisseur_excluded = []


cleaned_rows_current_year = []
cleaned_rows_past_year = []
current_fournisseur_number = None
current_fournisseur_name = None
start_date = datetime.datetime.strptime('2025-06-01', '%Y-%m-%d').date() 
end_date = datetime.datetime.strptime('2025-09-01', '%Y-%m-%d').date() 
start_year  = datetime.datetime.strptime('2025-01-01', '%Y-%m-%d').date() 
start_date_1 = start_date - datetime.timedelta(days=1)

df_raw_current_year = pd.read_excel('GRAND LIVRE FOURNISSEURS 2025.xlsx', header=None) 
for i, row in df_raw_current_year.iterrows():
        first_cell = str(row[0]).strip()

        if first_cell.startswith('4411'):
            current_fournisseur_number = first_cell.split()[0]
            current_fournisseur_name = ' '.join(first_cell.split()[1:])

        elif str(row[1]).strip() in JOURNAL_CODES:
            cleaned_rows_current_year.append({
                'Num fournisseur': current_fournisseur_number,
                'Nom fournisseur': current_fournisseur_name,
                'journal': row[1],
                'date': row[2].date() if pd.notna(row[2]) else None,
                'N° de pièce': row[3],
                'libelle': row[4],
                'montant_debit': row[8],
                'montant_credit': row[10],
                'lettrage': row[9]
            })

df_raw_past_year = pd.read_excel('Grand livre fournisseurs 2024 (SOMPAGRAF).xlsx', header=None) 
for i, row in df_raw_past_year.iterrows():
        first_cell = str(row[0]).strip()

        if first_cell.startswith('4411'):
            current_fournisseur_number = first_cell.split()[0]
            current_fournisseur_name = ' '.join(first_cell.split()[1:])

        elif str(row[1]).strip() in JOURNAL_CODES:
            cleaned_rows_past_year.append({
                'Num fournisseur': current_fournisseur_number,
                'Nom fournisseur': current_fournisseur_name,
                'journal': row[1],
                'date': row[2].date() if pd.notna(row[2]) else None,
                'N° de pièce': row[3],
                'libelle': row[4],
                'montant_debit': row[8],
                'montant_credit': row[10],
                'lettrage': row[9]
            })


df_results_current_year = pd.DataFrame(cleaned_rows_current_year)
df_clean = df_results_current_year[~df_results_current_year['Num fournisseur'].isin(fournisseur_excluded)]


df_results_past_year = pd.DataFrame(cleaned_rows_past_year)
df_clean_past = df_results_past_year[~df_results_past_year['Num fournisseur'].isin(fournisseur_excluded)]
df_clean_past = df_clean_past[df_clean_past['journal'] == 'HA']



# Facture trimestre paye / Non paye

In [84]:

factures_trimester = df_clean[
    (df_clean['journal'].isin(['HA'])) &
    (df_clean['montant_credit'].notna()) &
    (df_clean['montant_credit'] > 0)
    &( df_clean['date'].between(start_date, end_date) )
]
paiements_trimester = df_clean[
    (
        ((df_clean['journal'].isin(['BC', 'CA', 'AN'])) & (df_clean['montant_debit'].notna())) 
        |
        ((df_clean['journal'] == 'HA') & (df_clean['montant_credit'] < 0))
    )
    &
    ( df_clean['date'].between(start_date, end_date) )
]

paiements_trimester = paiements_trimester.copy()
paiements_trimester['montant_paiement'] = paiements_trimester['montant_debit'].fillna(paiements_trimester['montant_credit'].abs())

join = pd.merge(
    factures_trimester,
    paiements_trimester[['Num fournisseur', 'lettrage', 'montant_paiement', 'date']],
    on=['Num fournisseur', 'lettrage'],
    how='left',
    suffixes=('_facture', '_paiement')
)

result_query_1 = join
excel = join.to_excel('query1.xlsx', index=None)



# Past invoice paid in trimester

In [ ]:

factures_avant_trimester = df_clean[
    (df_clean['journal'].isin(['HA'])) &
    (df_clean['montant_credit'].notna()) &
    (df_clean['montant_credit'] > 0)
    &( df_clean['date'].between(start_year, start_date_1)  )
]
paiements_trimester = df_clean[
    (
        ((df_clean['journal'].isin(['BC', 'CA', 'AN'])) & (df_clean['montant_debit'].notna())) 
        |
        ((df_clean['journal'] == 'HA') & (df_clean['montant_credit'] < 0))
    )&
    ( df_clean['date'].between(start_date, end_date)  )
]

paiements_trimester = paiements_trimester.copy()
paiements_trimester['montant_paiement'] = paiements_trimester['montant_debit'].fillna(paiements_trimester['montant_credit'].abs())

join = pd.merge(
    factures_avant_trimester,
    paiements_trimester[['Num fournisseur', 'lettrage', 'montant_paiement', 'date']],
    on=['Num fournisseur', 'lettrage'],
    how='left',
    suffixes=('_facture', '_paiement')
)

result_query_2 = join[join['date_paiement'].notna() ]

result_query_2
# excel = result_query_2.to_excel('output.xlsx', index=None)



# Past invoice not paid

In [86]:
factures_avant_trimester = df_clean[
    (df_clean['journal'].isin(['HA'])) &
    (df_clean['montant_credit'].notna()) &
    (df_clean['montant_credit'] > 0) &
    (df_clean['date'].between(start_year, start_date_1))
]

paiements_avant_trimester = df_clean[
    (
        ((df_clean['journal'].isin(['BC', 'CA', 'AN'])) & (df_clean['montant_debit'].notna())) 
        |
        ((df_clean['journal'] == 'HA') & (df_clean['montant_credit'] < 0))
    ) &
    (df_clean['date'].between(start_year, end_date))
]
paiements_avant_trimester = paiements_avant_trimester.copy()
paiements_avant_trimester['montant_paiement'] = paiements_avant_trimester['montant_debit'].fillna(paiements_avant_trimester['montant_credit'].abs())

join = pd.merge(
    factures_avant_trimester,
    paiements_avant_trimester[['Num fournisseur', 'lettrage', 'montant_paiement', 'date']],
    on=['Num fournisseur', 'lettrage'],
    how='left',
    suffixes=('_facture', '_paiement')
)


result_query_3 = join[join['date_facture'].notna()]




In [87]:
factures_AN = df_clean[
    (df_clean['journal'] == 'AN') &
    (df_clean['montant_credit'].notna()) &
    (df_clean['montant_credit'] > 0) &
    (df_clean['date'] == start_year)
]

factures_HA = df_clean_past[
    (df_clean_past['journal'] == 'HA') &
    (df_clean_past['montant_credit'].notna()) &
    (df_clean_past['montant_credit'] > 0)
]
matched_AN = pd.merge(
    factures_AN,
    factures_HA[['Num fournisseur', 'N° de pièce', 'date', 'montant_credit']],
    on=['Num fournisseur', 'N° de pièce','montant_credit'],
    how='inner',
    suffixes=('_an', '_original')
)

paiements_trimester = df_clean[
    (
        ((df_clean['journal'].isin(['BC', 'CA', 'AN'])) & (df_clean['montant_debit'].notna())) |
        ((df_clean['journal'] == 'HA') & (df_clean['montant_credit'] < 0))
    ) &
    (df_clean['date'].between(start_date, end_date))
].copy()
paiements_trimester['montant_paiement'] = paiements_trimester['montant_debit'].fillna(paiements_trimester['montant_credit'].abs())


join = pd.merge(
    matched_AN,
    paiements_trimester[['Num fournisseur', 'N° de pièce', 'date', 'montant_paiement','lettrage']],
    on=['Num fournisseur', 'lettrage'],
    how='left',
    suffixes=('_an', '_paiement')
)

paid_AN = join[join['date'].notna()]


join.to_excel('Rn-Paid-trimester.xlsx', index=None)




# RN paid in trimester

In [88]:
factures_AN = df_clean[
    (df_clean['journal'] == 'AN') &
    (df_clean['montant_credit'].notna()) &
    (df_clean['montant_credit'] > 0) &
    (df_clean['date'] == start_year)
]

factures_HA = df_clean_past[
    (df_clean_past['journal'] == 'HA') &
    (df_clean_past['montant_credit'].notna()) &
    (df_clean_past['montant_credit'] > 0)
]
matched_AN = pd.merge(
    factures_AN,
    factures_HA[['Num fournisseur', 'N° de pièce', 'date', 'montant_credit']],
    on=['Num fournisseur', 'N° de pièce','montant_credit'],
    how='inner',
    suffixes=('_an', '_original')
)

paiements_trimester = df_clean[
    (
        ((df_clean['journal'].isin(['BC', 'CA', 'AN'])) & (df_clean['montant_debit'].notna())) |
        ((df_clean['journal'] == 'HA') & (df_clean['montant_credit'] < 0))
    ) &
    (df_clean['date'].between(start_date, end_date))
].copy()
paiements_trimester['montant_paiement'] = paiements_trimester['montant_debit'].fillna(paiements_trimester['montant_credit'].abs())


join = pd.merge(
    matched_AN,
    paiements_trimester[['Num fournisseur', 'N° de pièce', 'date', 'montant_paiement','lettrage']],
    on=['Num fournisseur', 'lettrage'],
    how='left',
    suffixes=('_an', '_paiement')
)

result_query_4 = join[join['date'].notna()]






# RN still not paid

In [89]:
factures_AN = df_clean[
    (df_clean['journal'] == 'AN') &
    (df_clean['montant_credit'].notna()) &
    (df_clean['montant_credit'] > 0) &
    (df_clean['date'] == start_year)
]

factures_HA = df_clean_past[
    (df_clean_past['journal'] == 'HA') &
    (df_clean_past['montant_credit'].notna()) &
    (df_clean_past['montant_credit'] > 0)
]
matched_AN = pd.merge(
    factures_AN,
    factures_HA[['Num fournisseur', 'N° de pièce', 'date', 'montant_credit']],
    on=['Num fournisseur', 'N° de pièce','montant_credit'],
    how='inner',
    suffixes=('_an', '_original')
)

paiements_trimester = df_clean[
    (
        ((df_clean['journal'].isin(['BC', 'CA', 'AN'])) & (df_clean['montant_debit'].notna())) |
        ((df_clean['journal'] == 'HA') & (df_clean['montant_credit'] < 0))
    ) &
    (df_clean['date'].between(start_year, end_date))
].copy()
paiements_trimester['montant_paiement'] = paiements_trimester['montant_debit'].fillna(paiements_trimester['montant_credit'].abs())


join = pd.merge(
    matched_AN,
    paiements_trimester[['Num fournisseur', 'N° de pièce', 'date', 'montant_paiement','lettrage']],
    on=['Num fournisseur', 'lettrage'],
    how='left',
    suffixes=('_an', '_paiement')
)

result_query_5 = join[join['date'].isna()]






In [ ]:
result = pd.concat([result_query_1, result_query_2, result_query_3, result_query_4, result_query_5])
result.to_excel('full_report.xlsx')